In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime

def get_adjusted_prices(tickers, start="2020-01-01", end=None):
    data = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
        group_by="column"
    )

    if isinstance(data.columns, pd.MultiIndex):
        # Multi-ticker download
        if "Adj Close" in data.columns.levels[0]:
            prices = data["Adj Close"].copy()
        elif "Close" in data.columns.levels[0]:
            prices = data["Close"].copy()
        else:
            raise ValueError("Could not find Close/Adj Close in downloaded data.")
    else:
        # Single ticker fallback
        if "Adj Close" in data.columns:
            prices = data[["Adj Close"]].copy()
        elif "Close" in data.columns:
            prices = data[["Close"]].copy()
        else:
            raise ValueError("Could not find Close/Adj Close in downloaded data.")
        prices.columns = tickers

    prices = prices.dropna()
    if prices.shape[1] != 2:
        raise ValueError("Expected exactly two tickers.")
    return prices

def get_closest_expiration(expirations, target_days=365):
    today = datetime.today()

    exp_dates = pd.to_datetime(expirations)
    days = [(exp - today).days for exp in exp_dates]

    # find closest to target_days
    idx = np.argmin([abs(d - target_days) for d in days])

    return expirations[idx]

def get_strike_price(ticker, option_type="put", target="ATM"):
    tk = yf.Ticker(ticker)

    # Use closest expiration
    exp = get_closest_expiration(tk.options, target_days=365)

    chain = tk.option_chain(exp)
    options = chain.puts if option_type == "put" else chain.calls

    # Current price
    S0 = tk.history(period="1d")["Close"].iloc[-1]

    # Find closest strike
    options["distance"] = np.abs(options["strike"] - S0)

    if target == "ATM":
        row = options.sort_values("distance").iloc[0]

    elif target == "OTM":
        # slightly out-of-the-money put (strike < price)
        otm = options[options["strike"] < S0]
        row = otm.sort_values("distance").iloc[0]

    else:
        raise ValueError("target must be 'ATM' or 'OTM'")

    return {
        "strike": float(row["strike"]),
        "price": float(row["lastPrice"]),
        "expiration": exp,
        "S0": float(S0)
    }

In [ ]:
def estimate_moments(prices, periods_per_year=252):
    log_returns = np.log(prices / prices.shift(1)).dropna()

    mu = log_returns.mean() * periods_per_year
    sigma = log_returns.std(ddof=1) * np.sqrt(periods_per_year)
    rho = log_returns.corr().iloc[0, 1]

    return {
        "mu1": float(mu.iloc[0]),
        "mu2": float(mu.iloc[1]),
        "sigma1": float(sigma.iloc[0]),
        "sigma2": float(sigma.iloc[1]),
        "rho": float(rho),
        "log_returns": log_returns
    }

In [ ]:
def build_three_states_from_moments(mu1, mu2, sigma1, sigma2, rho):
    # Stock 1 deviations
    d1 = sigma1 * np.sqrt(3 / 2)
    x = np.array([d1, 0.0, -d1])  # centered deviations for stock 1

    # Need y for stock 2 such that:
    # mean(y)=0
    # sum(y^2)/3 = sigma2^2
    # sum(x*y)/3 = rho*sigma1*sigma2
    #
    # Use y = a*x + z, where z is orthogonal to x and sums to zero.
    target_cov = rho * sigma1 * sigma2
    a = target_cov / sigma1**2

    # Basis vector orthogonal to x and summing to zero
    z_base = np.array([1.0, -2.0, 1.0])

    var_x = np.sum(x**2) / 3.0
    var_z = np.sum(z_base**2) / 3.0

    residual_var = sigma2**2 - a**2 * var_x
    if residual_var < -1e-12:
        raise ValueError("Target correlation too extreme for this 3-state construction.")
    residual_var = max(residual_var, 0.0)

    b = np.sqrt(residual_var / var_z)
    y = a * x + b * z_base

    r1_states = mu1 + x
    r2_states = mu2 + y

    return r1_states, r2_states

In [ ]:
def solve_risk_neutral_probs_from_returns(r1_states, r2_states, risk_free_rate):
    r1_states = np.asarray(r1_states, dtype=float)
    r2_states = np.asarray(r2_states, dtype=float)

    R1 = 1.0 + r1_states
    R2 = 1.0 + r2_states

    A = np.array([
        [1.0, 1.0, 1.0],
        R1,
        R2
    ], dtype=float)

    b = np.array([
        1.0,
        1.0 + risk_free_rate,
        1.0 + risk_free_rate
    ], dtype=float)

    q = np.linalg.solve(A, b)
    return q

In [ ]:
def run_real_data_two_stock_model(
    ticker1,
    ticker2,
    start="2020-01-01",
    end=None,
    risk_free_rate=0.05,
):
    prices = get_adjusted_prices([ticker1, ticker2], start=start, end=end)
    moments = estimate_moments(prices)

    mu1 = moments["mu1"]
    mu2 = moments["mu2"]
    sigma1 = moments["sigma1"]
    sigma2 = moments["sigma2"]
    rho = moments["rho"]

    # Build 3 real-world states from estimated moments
    r1_states, r2_states = build_three_states_from_moments(mu1, mu2, sigma1, sigma2, rho)

    # Solve risk-neutral probabilities
    q = solve_risk_neutral_probs_from_returns(r1_states, r2_states, risk_free_rate)

    # Current spot prices
    S1_0 = float(prices.iloc[-1, 0])
    S2_0 = float(prices.iloc[-1, 1])

    # $100 equal-dollar portfolio:
    # $50 in each stock
    w1 = 0.5
    w2 = 0.5
    total_capital = 100.0

    n1 = (w1 * total_capital) / S1_0
    n2 = (w2 * total_capital) / S2_0

    # Real-world probabilities
    p = np.array([1/3, 1/3, 1/3], dtype=float)

    # Terminal stock prices in each state
    S1_T = S1_0 * (1.0 + r1_states)
    S2_T = S2_0 * (1.0 + r2_states)

    # Unhedged terminal portfolio values
    unhedged_terminal = n1 * S1_T + n2 * S2_T

    # Strikes
    strike1_data = get_strike_price(ticker1)
    strike2_data = get_strike_price(ticker2)

    K1 = strike1_data["strike"]
    K2 = strike2_data["strike"]
    Kb = n1 * K1 + n2 * K2

    # Put payoffs
    put1_payoff = np.maximum(K1 - S1_T, 0.0)
    put2_payoff = np.maximum(K2 - S2_T, 0.0)

    two_put_payoff = n1 * put1_payoff + n2 * put2_payoff
    basket_put_payoff = np.maximum(Kb - unhedged_terminal, 0.0)

    # Option prices via risk-neutral probabilities
    disc = 1.0 / (1.0 + risk_free_rate)

    P1 = disc * np.dot(q, put1_payoff)
    P2 = disc * np.dot(q, put2_payoff)

    two_put_cost = n1 * P1 + n2 * P2
    basket_put_cost = disc * np.dot(q, basket_put_payoff)

    # Hedged terminal values
    two_put_terminal = unhedged_terminal + two_put_payoff
    basket_put_terminal = unhedged_terminal + basket_put_payoff

    # base stock investment = 100
    # only hedge cost is financed to the end of the period
    financed_cost_unhedged = 100.0
    financed_cost_two_put = 100.0 + (1.0 + risk_free_rate) * two_put_cost
    financed_cost_basket = 100.0 + (1.0 + risk_free_rate) * basket_put_cost

    def make_row(name, terminal_values, financed_cost):
        expected_terminal = np.dot(p, terminal_values)
        worst_case = np.min(terminal_values)
        expected_return = (expected_terminal - financed_cost) / 100.0
        return {
            "Strategy": name,
            "Financed Cost": financed_cost,
            "Expected Terminal Value": expected_terminal,
            "Worst-Case Value": worst_case,
            "Expected Return": expected_return
        }

    summary = pd.DataFrame([
        make_row("Unhedged", unhedged_terminal, financed_cost_unhedged),
        make_row("Two puts", two_put_terminal, financed_cost_two_put),
        make_row("Basket puts", basket_put_terminal, financed_cost_basket),
    ])

    detail = pd.DataFrame({
        "State": ["U", "M", "D"],
        "Real Prob": p,
        "RN Prob": q,
        "r1_state": r1_states,
        "r2_state": r2_states,
        "S1_T": S1_T,
        "S2_T": S2_T,
        "Unhedged": unhedged_terminal,
        "Two-put payoff": two_put_payoff,
        "Basket-put payoff": basket_put_payoff,
        "Two-puts strategy": two_put_terminal,
        "Basket-put strategy": basket_put_terminal,
    })

    return {
        "summary": summary,
        "detail": detail,
        "moments": moments,
        "spots": {"S1_0": S1_0, "S2_0": S2_0},
        "strikes": {"K1": K1, "K2": K2, "Kb": Kb},
        "option_prices": {
            "P1": P1,
            "P2": P2,
            "two_put_cost": two_put_cost,
            "basket_put_cost": basket_put_cost
        },
        "risk_neutral_probs": q
    }

In [ ]:
result = run_real_data_two_stock_model(
    ticker1="AAPL",
    ticker2="MSFT",
    start="2022-01-01",
    risk_free_rate=0.05
)

summary = result["summary"].copy()
summary["Financed Cost"] = summary["Financed Cost"].round(2)
summary["Expected Terminal Value"] = summary["Expected Terminal Value"].round(2)
summary["Worst-Case Value"] = summary["Worst-Case Value"].round(2)
summary["Expected Return"] = (100 * summary["Expected Return"]).round(2).astype(str) + "%"

print("\nRisk-neutral probabilities:")
print(result["risk_neutral_probs"])

print("\nSummary table:")
print(summary.to_string(index=False))